In [ ]:
# get the code
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# use seaborn plotting defaults
import seaborn as sns; sns.set()
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle

# import package functions (main: sys.path.append('../code') and the modules in code/)
import mycnv
from mycnv.rolling import interpolate, one_col_rolling, llh, rolling_data
from mycnv.rolling_cov import rolling_coverage
from mycnv.rolling_snp import filter_snp
from mycnv.plot import plot_cov, plot_snp

# main: pd.set_option('mode.chained_assignment', None); no effect with pandas >= 3 (copy-on-write)

# paths and parameters: ../config/config_dev.yaml via dev_config.py (main: hard-coded Dropbox paths)
from dev_config import load_dev
dev = load_dev()
plot_path = dev["plot_path"]
fig_path = dev["fig_path"]
print(f"mycnv {mycnv.__version__} ({os.path.dirname(mycnv.__file__)})")
print(f"sample {dev['sample']} ({dev['study']}) -> {dev['out_dir']}")

## visualize coverage data

In [ ]:
sample = dev["sample"]
# main read output/CNV/{sample}.cnv.full.gz, a name notebook 4 never wrote (it writes .cnv.gz)
cnv_df = pd.read_csv(dev["chain_cnv"], sep="\t")
rsnp_df = pd.read_csv(dev["chain_cnv_snp"], sep="\t")
cnv_df[:3]
rsnp_df[:3]

### load helpers

In [ ]:
# canonical code (was an inline copy of code/plot_helpers.py): mycnv.plot_helpers
# changes vs main: matplotlib.colormaps (plt.cm.get_cmap is gone in matplotlib >= 3.9), natural chromosome order,
# .iloc instead of Series[-1], one rectangle per chromosome present
from mycnv.plot_helpers import (sort_df, get_chrom_df, make_color_chroms, add_chrom_labels, make_nice,
                                get_tick_pos, str_pos, get_precision, set_ticks, extract_pos)

## coverage plotting

In [ ]:
from mycnv.plot import plot_cov   # was an inline copy of code/plot.py:plot_cov

In [ ]:
log2 = dict(
        title='L2R',
        plot_type='scatter',   # ['line', 'scatter']
        data='log2ratio2',
        plot_args=dict(
            linewidth=0.3,
            color='black',
            s=0.8,
            alpha=.7
        )
    )

log2mean = dict(
        title='L2R_mean',
        plot_type='line',   # ['line', 'scatter']
        data='log2ratio2_mean',
        plot_args=dict(
            linewidth=1,
            color='yellow',
            alpha=.7
        )
    )

chroms = ['chr5', 'chr7','chr8', 'chr11', 'chr17']

fig_params = dict(
    figsize=(34,4),
    colormap='coolwarm_r',
    color_chroms=True,
    ylim=(-1.5,2.5),
    label_size=15
)
chroms = ['chr3', 'chr4', 'chr5', 'chr6','chr7', 'chr9', 'chr12', 'chr17']
r20 = 'chr20:20.7Mb-34.5Mb'
r7 = 'chr7:95.8Mb-111Mb'
r17 = 'chr17:18.2Mb-25Mb'
fig, ax, df, chrom_df = plot_cov(cnv_df, plots=[log2, log2mean], chroms="all", region= 'chr7:95.8Mb-110Mb', **fig_params)

In [ ]:
fig, ax, df, chrom_df = plot_cov(cnv_df.query('log2ratio2 == log2ratio2'), plots=[log2, log2mean], chroms="all", region='', **fig_params)

# visualize heteroSNP

sample = dev["sample"]
snp_df = pd.read_csv(dev["raw_snp"], sep="\t")   # raw/{sample}.snp is plain text

In [ ]:
snp_df = cnv_df.query("VAF2 == VAF2")
snp_df[:3]

### for hetSNP visualization:
    + snp_vis in specified by snp_plots

In [ ]:
from mycnv.plot import plot_snp   # was an inline copy of code/plot.py:plot_snp

In [ ]:
vaf = dict(
        title='VAF',
        plot_type='scatter',  # ['line', 'scatter']
        data='VAF2',
        plot_args=dict(
            s=2,
            color='black',
            alpha=1
        )
    )

fig_params = dict(
    figsize=(34,4),
    colormap='coolwarm_r',
    color_chroms=True,
    ylim=(0,1),
    label_size=13
)

fig, ax, df, chrom_df = plot_snp(snp_df, plots=[vaf], chroms="all", region='', **fig_params)

# combine SNP and coverage figs

In [ ]:
from mycnv.plot import plot_CNV   # was an inline copy of code/plot.py:plot_CNV

In [ ]:
log2 = dict(
        title='L2R',
        plot_type='scatter',   # ['line', 'scatter']
        data='log2ratio2',
        plot_args=dict(
            linewidth=0.3,
            color='black',
            s=1.8,
            alpha=1
        )
    )

log2mean = dict(
        title='L2R_mean',
        plot_type='line',   # ['line', 'scatter']
        data='log2ratio2_mean',
        plot_args=dict(
            linewidth=1,
            color='yellow',
            alpha=.7
        )
    )

vaf = dict(
        title='VAF',
        plot_type='scatter',  # ['line', 'scatter']
        data='VAF2',
        plot_args=dict(
            s=2,
            color='black',
            alpha=1
        )
    )

fig_params = dict(
    figsize=(34,6),
    colormap='coolwarm_r',
    color_chroms=True,
    label_size=13,
    ylims=dict(
        cov=(-1.4,2.5),
        snp=(0, 1)
    )
)
chroms = ['chr3', 'chr4', 'chr5', 'chr6','chr7', 'chr9', 'chr12', 'chr17']
r20 = 'chr20:20.7Mb-34.5Mb'
r7 = 'chr7:95.8Mb-111Mb'
r17 = 'chr17:18.2Mb-25Mb'

fig, axes, df, chrom_df = plot_CNV(cnv_df, cov_plots=[log2, log2mean], snp_plots=[vaf], chroms="all", region="chr20", **fig_params)

# plot for SNP matching

In [ ]:
from mycnv.plot import make_SNP_plot   # was an inline copy of code/plot.py:make_SNP_plot

In [ ]:
sample = dev["sample"]
cnv_df = pd.read_csv(dev["chain_cnv_snp"], sep="\t")
_ = make_SNP_plot(sample, cnv_df)

# visualize clusterings

### load example clustered file from DBscan clustering
+ Gen-1 artefact (`output/cluster/*.dbscan.csv`, written by CNVinspect). None exists for the study data, so the
  cells below skip the plot when the file is missing.

In [ ]:
sample = "01_A"
dbscan_file = os.path.join(dev["out_dir"], f"cluster/{sample}.dbscan.csv")
df = pd.read_csv(dbscan_file, sep='\t') if os.path.isfile(dbscan_file) else None
print("cluster file", dbscan_file, "found" if df is not None else "not found: cluster plot skipped")

In [ ]:
df

In [ ]:
log2 = dict(
        title='log2ratio',
        plot_type='scatter',   # ['line', 'scatter']
        data='log2ratio',
        plot_args=dict(
            linewidth=0.3,
            color='black',
            s=0.2,
            alpha=.7
        )
    )

log2mean = dict(
        title='rollinglog2ratio',
        plot_type='line',   # ['line', 'scatter']
        data='log2ratiomean',
        plot_args=dict(
            linewidth=1,
            color='yellow',
            alpha=.7
        )
    )

absvaf = dict(
        title='absVAF',
        plot_type='scatter',  # ['line', 'scatter']
        data='absVAF',
        plot_args=dict(
            s=1,
            c='dbscan',
            alpha=.7
        )
    )

absvafmean = dict(
        title='deltaVAFvar',
        plot_type='line',   # ['line', 'scatter']
        data='deltaVAFvar',
        plot_args=dict(
            linewidth=1,
            color='blue',
            alpha=.7
        )
    )

fig_params = dict(
    figsize=(24,8),
    colormap='coolwarm_r',
    color_chroms=False,
    ylim=(0,1),
    cov_offset=.1,  # how much log2ratio=0 is shifted above SNP-data
    cov_height=.5,
    label_size=13
)


### expand plot_snp functionality to showing clusters

In [ ]:
# Gen-1 plot_snp with a coverage overlay (renamed: it shadowed plot_snp)
def plot_snp_cluster(df, snp_plots=[], cov_plots=[], chroms='all', cov_offset=.25, cov_height=0.5, color_chroms=True, colormap='coolwarm_r', region='', label_size=12, figsize=(20,4), ylim=(-1,1)):
    
    MAXLOG2RATIO = 2.5
    #### DATA MANGELING ##########
    # get cols for rearranging
    org_cols = list(df.columns)
    
    # sort the df
    df = sort_df(df)
    # reduce the df to the selected chromosomes
    if region:
        chrom, start, end = extract_pos(region)
        df = df.query('Chr == @chrom and @start <= Pos <= @end')
    elif chroms != 'all':
        df = df.query('Chr in @chroms')

    # get the chrom_df for collapsing the 
    chrom_df = get_chrom_df(df)
    
    df = df.merge(chrom_df.loc[:, 'dif'], on='Chr')
    df['PlotPos'] = df['FullExonPos'] - df['dif']
    
    # rearrange the df as return value
    new_cols = org_cols[:4] + ['PlotPos'] + org_cols[4:]
    df = df.loc[:, new_cols]
    
    #########################
    ######## PLOTTING #######
    # plot the figure
    fig, ax = plt.subplots(figsize=figsize)

    # set the x-axis limits
    _ = ax.set_xlim(0, df['PlotPos'].max())
    
    
    ######## PLOT COV Data
    
    if len(cov_plots):
        scale_factor = cov_height / (MAXLOG2RATIO + 1)
        offset = 1 + scale_factor + cov_offset

        ylim = (ylim[0], ylim[1] + cov_offset + cov_height)
        
        for plot in cov_plots:
            # normalize the coverage data:
            # 2.5 is the approx max log2ratio (LOH to 8N)

            df[plot['data']] = df[plot['data']] * scale_factor + offset
            if plot['plot_type'] == 'line':
                plot = ax.plot(df['PlotPos'],df[plot['data']], **plot['plot_args'])
            elif plot['plot_type'] == 'scatter':
                # highjack plot_args
                pa = plot['plot_args']
                if "c" in pa:
                    pa['c'] = df[pa['c']]
                if "s" in pa:
                    if isinstance(pa['s'], str):
                        pa['s'] = df[pa['s']] * 20 + 1
                plot = ax.scatter(df['PlotPos'],df[plot['data']], **pa)
    
    ######## plot the SNP graphs #######
    for plot in snp_plots:
        if plot['plot_type'] == 'line':
            plot = ax.plot(df['PlotPos'],df[plot['data']], **plot['plot_args'])
        elif plot['plot_type'] == 'scatter':
            # highjack plot_args
            pa = plot['plot_args']
            if "c" in pa:
                    pa['c'] = df[pa['c']]
            if "s" in pa:
                if isinstance(pa['s'], str):
                    pa['s'] = df[pa['s']] * 20 + 1
            plot = ax.scatter(df['PlotPos'],df[plot['data']], **pa)
    
    _ = ax.set_ylim(ylim)
    # add the color chroms
    _ = make_color_chroms(ax, chrom_df, color_chroms, ylimits=ax.get_ylim(), colormap=colormap)
    
    
    ######## LABELS ###################
    # set the axis labels
    _ = ax.set_xlabel('genomic coords', fontsize=1.25*label_size);
    # quick fix for one y-label
    _ = ax.set_ylabel(' / '.join([plot['title'] for plot in snp_plots]), fontsize=1.25*label_size)
    
    ######## CHROM LABELS #############
    add_chrom_labels(ax, chrom_df, ax.get_ylim())
    
    ####### X-AXIS ####################
    # set major ticks and grid for chrom
    
    ax = set_ticks(ax, df, chrom_df, label_size=label_size)
    
    # return fig and ax for further plotting and return edited dataframe
    return fig, ax, df, chrom_df

In [ ]:
absvaf = dict(
        title='absVAF',
        plot_type='scatter',  # ['line', 'scatter']
        data='absVAF',
        plot_args=dict(
            c='dbscan',
            s='dbscan',
            alpha=.5,
            cmap='rainbow'
        )
    )

if df is not None:
    _ = plot_snp_cluster(df, snp_plots=[absvaf, absvafmean], cov_plots=[log2, log2mean], chroms=chroms, region='chr17', **fig_params)